# ATLAS Solar Scenario Data Download Tutorial

This notebook downloads CMIP6 scenario data from ESGF and stores the files using a portable folder structure.
Documentation: https://esgf.github.io/esgf-user-support/metagrid.html

The notebook is organised as a tutorial for users who may not be experienced with Python:

1. **Set input parameters**: choose the country, model, experiment, variable and time range.
2. **Load helper functions**: functions used to query ESGF and download files.
3. **Run the download**: execute the workflow and save NetCDF files locally.

The output folder follows this convention:

```text
../data/esgf_download/{model}/{experiment}/{variable}/{country}/
```

Example:

```text
../data/esgf_download/CNRM-ESM2-1/ssp370/rivo/italy/
```


## Step 1: Set the input parameters

Edit only the values in this cell before running the notebook.

Parameter meaning:

- `country`: country name used only to organise the output folders. The ESGF download itself is not clipped by country.
- `model`: CMIP6 model name.
- `experiments`: list of CMIP6 experiments to download. For example `historical`, `ssp245`, `ssp370`, `ssp585`.
- `variable`: CMIP6 variable name to download. Documentation: https://cmip6dr.github.io/Data_Request_Home/
- `table`: CMIP6 table identifier, for example `Eday`.
- `variant_preference`: preferred CMIP6 ensemble members. The notebook tries them in order and uses the first available one.
- `search_node`: ESGF search node.
- `download_root`: root folder where all downloaded ESGF files will be stored.


In [1]:
from pathlib import Path

# Country used to organise the local output folder.
country = None  # Downloading "global"

# CMIP6 download parameters.
model = "CNRM-ESM2-1"
experiments = ["historical", "ssp370"]  # examples: ["historical", "ssp370", "ssp585"]
variable = "rsds"              # examples: "tas", "rsds", "clt"
table = "day"

# Preferred ensemble members. The first available variant will be selected.
variant_preference = ["r1i1p1f1", "r1i1p1f2"]

# ESGF search nodes. The notebook tries them in order.
search_nodes = [
    "https://esgf-node.llnl.gov/esg-search",
    "https://esgf-data.dkrz.de/esg-search",
]

# Use distributed search to discover replicas on multiple ESGF nodes.
use_distributed_search = True

# Portable output root. Final files are saved as:
# ../data/esgf_downloads/{model}/{experiment}/{variable}/
download_root = Path("../data/esgf_downloads")

# Download settings.
chunk_size = 1 << 20  # 1 MiB

# Robust download settings.
download_timeout = 300       # seconds allowed without establishing/receiving data
max_retries = 3              # attempts for each URL
retry_sleep = 20             # initial seconds between attempts
skip_existing = True         # do not download files already present
remove_partial_files = True  # delete incomplete .part files after failed attempts


## Step 2: Load libraries and helper functions

Run this cell once. It defines the functions used by the download workflow.


In [2]:
import re
import time
import requests
from pathlib import Path
from pyesgf.search import SearchConnection


def normalise_name(value: str) -> str:
    """Return a simple folder-safe name."""
    return str(value).strip().lower().replace(" ", "_")


def normalise_url(url: str) -> str:
    """Clean ESGF URLs and force HTTPS when possible."""
    url = str(url).strip()
    if url.startswith("http://"):
        url = "https://" + url[len("http://"):]
    return url


def get_year_range(experiment: str) -> tuple[int, int]:
    """Return the default download period for historical or future CMIP6 experiments."""
    if experiment == "historical":
        return 1950, 2014
    return 2015, 2100


def extract_year_bounds(filename: str):
    """
    Extract start and end year from a CMIP6 filename.

    Example:
    tas_day_CNRM-ESM2-1_historical_r1i1p1f2_gr_18500101-20141231.nc
    """
    match = re.search(r"_(\d{4})\d{4}-(\d{4})\d{4}\.nc$", filename)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None


def build_output_dir(download_root: Path, model: str, experiment: str, variable: str, country: str | None) -> Path:
    """Build the output directory."""
    # If you want one folder per country, replace the next line with:
    # return download_root / model / experiment / variable / normalise_name(country)
    return download_root / model / experiment / variable


def get_http_urls(file_result):
    """
    Return all HTTPServer/fileServer URLs available for an ESGF file result.
    Some nodes expose multiple services. This function keeps only direct file URLs.
    """
    urls = []

    for item in getattr(file_result, "urls", []) or []:
        parts = item.split("|")
        url = normalise_url(parts[0])
        service = "|".join(parts[1:]) if len(parts) > 1 else ""

        if "HTTPServer" in service or "fileServer" in url:
            urls.append(url)

    download_url = getattr(file_result, "download_url", None)
    if download_url:
        urls.append(normalise_url(download_url))

    # Remove duplicates while preserving order.
    return list(dict.fromkeys(urls))


def fetch_file(
    url: str,
    destination: Path,
    chunk_size: int = 1 << 20,
    timeout: int = 300,
    max_retries: int = 3,
    retry_sleep: int = 20,
    skip_existing: bool = True,
    remove_partial_files: bool = True,
) -> bool:
    """Download one file from one URL, with retries and partial-file cleanup."""
    url = normalise_url(url)
    destination = Path(destination)

    if skip_existing and destination.exists() and destination.stat().st_size > 0:
        print(f"Already exists: {destination.name}")
        return True

    part_destination = destination.with_suffix(destination.suffix + ".part")

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Downloading: {destination.name} | attempt {attempt}/{max_retries}")

            with requests.get(url, stream=True, timeout=timeout) as response:
                response.raise_for_status()

                with part_destination.open("wb") as file_handle:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            file_handle.write(chunk)

            part_destination.replace(destination)
            print(f"Downloaded: {destination.name}")
            return True

        except requests.exceptions.ConnectTimeout as error:
            print(f"Connection timeout for {destination.name}: {error}")
        except requests.exceptions.ReadTimeout as error:
            print(f"Read timeout for {destination.name}: {error}")
        except requests.exceptions.ConnectionError as error:
            print(f"Connection error for {destination.name}: {error}")
        except requests.exceptions.HTTPError as error:
            print(f"HTTP error for {destination.name}: {error}")
            # 404/403 are usually not solved by retrying the same URL.
            break
        except Exception as error:
            print(f"Unexpected error for {destination.name}: {error}")

        if attempt < max_retries:
            wait_seconds = retry_sleep * attempt
            print(f"Waiting {wait_seconds} seconds before retrying...")
            time.sleep(wait_seconds)

    if remove_partial_files and part_destination.exists():
        part_destination.unlink()

    print(f"Failed from this URL: {destination.name}")
    return False


def make_search_nodes(search_nodes):
    """Accept either a string or a list of ESGF search nodes."""
    if isinstance(search_nodes, str):
        return [search_nodes]
    return list(search_nodes)


def find_datasets(
    search_nodes,
    model: str,
    experiment: str,
    variable: str,
    table: str,
    variant_preference: list[str],
    use_distributed_search: bool = True,
):
    """
    Search ESGF datasets for the requested variable.
    Returns all dataset replicas found, ordered by variant preference and search node.
    """
    datasets = []
    seen = set()

    for node in make_search_nodes(search_nodes):
        print(f"Searching ESGF node: {node}")
        try:
            conn = SearchConnection(node, distrib=use_distributed_search)
        except Exception as error:
            print(f"Could not connect to search node {node}: {error}")
            continue

        for variant in variant_preference:
            try:
                context = conn.new_context(
                    project="CMIP6",
                    source_id=model,
                    experiment_id=experiment,
                    variable_id=variable,
                    table_id=table,
                    variant_label=variant,
                    latest=True,
                    facets="project,activity_id,institution_id,source_id,experiment_id,variant_label,table_id,variable_id,grid_label",
                )
                results = context.search(ignore_facet_check=True)
            except Exception as error:
                print(f"Search failed for variant {variant} on {node}: {error}")
                continue

            for dataset in results:
                dataset_id = dataset.dataset_id
                if dataset_id not in seen:
                    seen.add(dataset_id)
                    datasets.append(dataset)

            if datasets:
                print(f"Found {len(datasets)} dataset replica(s) so far for variant {variant}.")
                # Stop at the first variant that exists, but keep all its replicas.
                return datasets

    return datasets


def list_files_in_year_range(dataset, year_start: int, year_end: int):
    """
    Return NetCDF files whose time span overlaps the selected period.

    This fixes historical tas, where the file can be 18500101-20141231.nc
    and would otherwise be excluded when requesting 1950-2014.
    """
    selected_files = []

    try:
        file_results = dataset.file_context().search(ignore_facet_check=True)
    except Exception as error:
        print(f"Could not list files for dataset replica: {dataset.dataset_id}")
        print(error)
        return selected_files

    for file_result in file_results:
        filename = file_result.filename

        if not filename.endswith(".nc"):
            continue

        file_start_year, file_end_year = extract_year_bounds(filename)

        if file_start_year is None or file_end_year is None:
            continue

        overlaps = file_start_year <= year_end and file_end_year >= year_start

        if overlaps:
            selected_files.append(file_result)

    return selected_files


The helper cell above already includes the corrected year-overlap logic and `ignore_facet_check=True` for ESGF file searches.

## Step 3: Run the ESGF download

This cell loops through all selected experiments, searches for the requested model and variable, filters files by year, and downloads the matching NetCDF files.


In [3]:
failed_downloads = []

for experiment in experiments:
    year_start, year_end = get_year_range(experiment)
    print(f"Experiment: {experiment}")
    print(f"Year range: {year_start}-{year_end}")

    datasets = find_datasets(
        search_nodes=search_nodes,
        model=model,
        experiment=experiment,
        variable=variable,
        table=table,
        variant_preference=variant_preference,
        use_distributed_search=use_distributed_search,
    )

    if not datasets:
        print("No matching dataset found. Skipping this experiment.")
        continue

    output_dir = build_output_dir(
        download_root=download_root,
        model=model,
        experiment=experiment,
        variable=variable,
        country=country,
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output folder: {output_dir}")

    downloaded_files = set()
    candidate_files = set()

    for dataset_index, dataset in enumerate(datasets, start=1):
        print(f"\nReplica {dataset_index}/{len(datasets)}")
        print(f"Dataset: {dataset.dataset_id}")

        files = list_files_in_year_range(dataset, year_start, year_end)

        if not files:
            print("No NetCDF files found in the selected time range for this replica.")
            continue

        print(f"Files matching selected period: {len(files)}")

        for file_result in files:
            filename = file_result.filename
            candidate_files.add(filename)

            if filename in downloaded_files:
                continue

            destination = output_dir / filename

            if skip_existing and destination.exists() and destination.stat().st_size > 0:
                print(f"Already exists, skipped: {filename}")
                downloaded_files.add(filename)
                continue

            urls = get_http_urls(file_result)
            print(f"\nAvailable HTTP URLs for {filename}: {len(urls)}")

            if not urls:
                print(f"No HTTP URL available for {filename}")
                continue

            success = False

            for url_index, url in enumerate(urls, start=1):
                print(f"\nURL {url_index}/{len(urls)}")
                print(f"Trying URL: {url}")

                success = fetch_file(
                    url=url,
                    destination=destination,
                    chunk_size=chunk_size,
                    timeout=download_timeout,
                    max_retries=max_retries,
                    retry_sleep=retry_sleep,
                    skip_existing=skip_existing,
                    remove_partial_files=remove_partial_files,
                )

                if success:
                    downloaded_files.add(filename)
                    break

            if not success:
                print(f"Failed download from this replica: {filename}")

    missing_files = sorted(candidate_files - downloaded_files)

    print(f"\nCompleted experiment. Files saved in: {output_dir}")

    if missing_files:
        print("Some files were not downloaded. You can rerun this cell later; existing files will be skipped.")
        for filename in missing_files:
            print(f"- {experiment} | {filename}")
            failed_downloads.append({"experiment": experiment, "filename": filename})
    else:
        print("All selected files were downloaded successfully.")

if failed_downloads:
    print("\nSummary of failed downloads:")
    for item in failed_downloads:
        print(f"- {item['experiment']} | {item['filename']}")
else:
    print("\nAll requested experiments completed successfully.")


Experiment: historical
Year range: 1950-2014
Searching ESGF node: https://esgf-node.llnl.gov/esg-search
Found 3 dataset replica(s) so far for variant r1i1p1f2.
Output folder: ../data/esgf_downloads/CNRM-ESM2-1/historical/clt

Replica 1/3
Dataset: CMIP6.CMIP.CNRM-CERFACS.CNRM-ESM2-1.historical.r1i1p1f2.day.clt.gr.v20181206|esgf-node.ornl.gov
Files matching selected period: 1

Available HTTP URLs for clt_day_CNRM-ESM2-1_historical_r1i1p1f2_gr_19500101-20141231.nc: 1

URL 1/1
Trying URL: https://esgf-node.ornl.gov/thredds/fileServer/css03_data/CMIP6/CMIP/CNRM-CERFACS/CNRM-ESM2-1/historical/r1i1p1f2/day/clt/gr/v20181206/clt_day_CNRM-ESM2-1_historical_r1i1p1f2_gr_19500101-20141231.nc
Downloading: clt_day_CNRM-ESM2-1_historical_r1i1p1f2_gr_19500101-20141231.nc | attempt 1/3
Downloaded: clt_day_CNRM-ESM2-1_historical_r1i1p1f2_gr_19500101-20141231.nc

Replica 2/3
Dataset: CMIP6.CMIP.CNRM-CERFACS.CNRM-ESM2-1.historical.r1i1p1f2.day.clt.gr.v20181206|eagle.alcf.anl.gov
Files matching selected per